# 10-bob. Anomaliyalarni aniqlash
## Amaliy misol: umumiy chegara va shaxsiy odatiy holat

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/10-bob/10-3_umumiy-chegara-va-shaxsiy-odatiy.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 10-bob.*

Quyidagi dasturda 20 nafar xodimning 60 kunlik faoliyati sun'iy ravishda yaratiladi: ochilgan fayllar soni, yuklab olingan ma'lumotlar hajmi va ish vaqtidan tashqari kirishlar soni. Har bir xodimning o'z odatiy darajasi bor. Ular orasida ikki alohida xodim mavjud: kuniga taxminan 2 gigabayt ma'lumot bilan ishlaydigan ma'lumotlar muhandisi (xodim_07) va ishdan ketishga tayyorlanib, oxirgi 10 kunda asta-sekin ko'proq ma'lumot nusxalay boshlagan xodim (xodim_13). Ikki yondashuv taqqoslanadi: hamma uchun bitta umumiy chegara va har bir xodimni o'zining dastlabki 50 kunlik odatiy holati bilan solishtirish. Ikkinchi yondashuvda har bir belgi uchun og'ish standart og'ishlar sonida (z-ball) o'lchanadi va musbat og'ishlar yig'indisi xavf bali sifatida olinadi. Dastur amaliy mashg'ulotning A10 mavzusiga mos keladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(4)
xodimlar = [f"xodim_{i:02d}" for i in range(1, 21)]
yozuvlar = []
for x in xodimlar:
    # Har bir xodimning o'z odatiy darajasi bor: kimdir ko'p fayl bilan ishlaydi, kimdir kam
    fayl_ortacha = rng.uniform(20, 120)
    mb_ortacha = 2000 if x == "xodim_07" else rng.uniform(50, 300)   # xodim_07 ma'lumotlar muhandisi
    for kun in range(1, 61):
        fayllar = rng.poisson(fayl_ortacha)
        mb = max(0, rng.normal(mb_ortacha, mb_ortacha * 0.25))
        tungi = rng.poisson(0.2)                                     # ish vaqtidan tashqari kirishlar
        if x == "xodim_13" and kun > 50:                             # ishdan ketishga tayyorlanayotgan xodim
            fayllar += 15 * (kun - 50)
            mb += 120 * (kun - 50)
            tungi += rng.poisson(2)
        yozuvlar.append((x, kun, fayllar, mb, tungi))

df = pd.DataFrame(yozuvlar, columns=["xodim", "kun", "fayllar", "mb", "tungi"])
belgilar = ["fayllar", "mb", "tungi"]
oxirgi = df[df.kun > 50]

In [2]:
# 1. Hamma uchun bitta umumiy chegara: kuniga 1000 MB dan ko'p ma'lumot olish
umumiy = oxirgi[oxirgi.mb > 1000].xodim.value_counts()
print("Umumiy chegara bo'yicha signallar (oxirgi 10 kun):")
print(umumiy.to_string(), "\n")

Umumiy chegara bo'yicha signallar (oxirgi 10 kun):
xodim
xodim_07    10
xodim_13     5 



In [3]:
# 2. Har bir xodimni o'zining 1-50-kunlardagi odatiy holati bilan solishtirish
asos = df[df.kun <= 50].groupby("xodim")[belgilar].agg(["mean", "std"])
def z_ball(qator):
    o, s = asos.loc[qator.xodim, (belgilar, "mean")].values, asos.loc[qator.xodim, (belgilar, "std")].values
    return pd.Series((qator[belgilar].values.astype(float) - o) / (s + 1e-6), index=belgilar)

z = oxirgi.apply(z_ball, axis=1)
oxirgi = oxirgi.assign(xavf=z.clip(lower=0).sum(axis=1))
reyting = oxirgi.groupby("xodim").xavf.mean().sort_values(ascending=False)
print("Shaxsiy odatiy holatga nisbatan o'rtacha xavf bali (eng yuqori 5 ta):")
print(reyting.head().round(1).to_string(), "\n")

print("xodim_13 ning 51-60-kunlardagi xavf bali:",
      oxirgi[oxirgi.xodim == "xodim_13"].xavf.round(1).tolist())

Shaxsiy odatiy holatga nisbatan o'rtacha xavf bali (eng yuqori 5 ta):
xodim
xodim_13    29.8
xodim_16     2.2
xodim_14     2.2
xodim_04     2.0
xodim_09     2.0 

xodim_13 ning 51-60-kunlardagi xavf bali: [13.2, 15.1, 18.1, 17.4, 21.2, 31.3, 36.8, 41.6, 45.0, 57.9]


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Umumiy chegara bo'yicha signallar (oxirgi 10 kun):
xodim
xodim_07    10
xodim_13     5

Shaxsiy odatiy holatga nisbatan o'rtacha xavf bali (eng yuqori 5 ta):
xodim
xodim_13    29.8
xodim_16     2.2
xodim_14     2.2
xodim_04     2.0
xodim_09     2.0

xodim_13 ning 51-60-kunlardagi xavf bali: [13.2, 15.1, 18.1, 17.4, 21.2, 31.3, 36.8, 41.6, 45.0, 57.9]
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 10-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.